In [ ]:
!pip install torchinfo
!pip install torchmetrics

In [ ]:
import os
import random
from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from PIL import Image
from typing import List
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import json # To save metrics
from tqdm.auto import tqdm


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Section 1

In [ ]:
def walk_through_dir(dir_path):
    for dirpath, dirname, filenames in os.walk(dir_path):
        print(f"There are {len(dirname)} directories and {len(filenames)} images in '{dirpath}'")


dataset_path = "MRI dataset"
walk_through_dir(dataset_path)

In [ ]:
train_dir = os.path.join(dataset_path, "train")
val_dir   = os.path.join(dataset_path, "val")

In [ ]:
from PIL import Image

train_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),   # <--- ensures 3 channels always
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),

    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


val_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
])

In [ ]:
train_dataset = datasets.ImageFolder(train_dir, transform = train_transform)
val_dataset   = datasets.ImageFolder(val_dir, transform = val_transform)

print("Original class mapping:", train_dataset.class_to_idx)
print("---------------------------------------------------------")
print (train_dataset)
print("=====================================================")
print (val_dataset)

In [ ]:
batch_size = 32
n_workers = 0

train_loader = DataLoader(train_dataset,batch_size=batch_size, shuffle=True, num_workers = n_workers)

val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of validation samples: {len(val_dataset)}")

print("============================================")

print(f"length of train_loader : {len(train_loader)} batches of  {batch_size}")
print(f"length of val_loader   : {len(val_loader)} batches of  {batch_size}")

# Section #2

In [ ]:
num_classes = 1  # binary classification
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False


model.fc = nn.Sequential(
    nn.Flatten(),
    nn.Dropout(0.3),
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, num_classes)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


In [ ]:
from torchinfo import summary
summary(model, input_size=(1, 3, 224, 224))

In [ ]:
from torchsummary import summary
summary(model, (3, 224, 224))

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
from timeit import default_timer as timer
def print_train_time (start : float,
                      end : float,
                      device : torch.device = None):

    total_time = end - start
    print(f"Total training time on {device}: {total_time:.3f} seconds")
    return total_time

In [ ]:
dummy_input = torch.randn(4, 3, 224, 224).to(device)
print(model(dummy_input).shape)  # Should be: torch.Size([4, 1]) to indecate binary classification

In [ ]:
print("Original class mapping:", train_dataset.class_to_idx)

In [ ]:
torch.manual_seed(42)
start_time = timer() # start timing
pos = 1
num_epochs = 20
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []
train_precisions, train_recalls, train_f1s = [], [], []
val_precisions, val_recalls, val_f1s = [], [], []
#%---------------------------- Training Loop ----------------------------
for epoch in range(num_epochs):
    model.train()

    running_loss, correct, total = 0.0, 0, 0

    all_labels, all_preds = [], []


    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]"):
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)

        labels = labels.float().unsqueeze(1)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        probs = torch.sigmoid(outputs)
        predicted = (probs > 0.5).long()

        running_loss += loss.item() * inputs.size(0)
        correct += (predicted == labels.long()).sum().item()
        total += labels.size(0)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(predicted.cpu().numpy())

    train_loss = running_loss / len(train_dataset)
    train_acc = correct / total
    train_precision = precision_score(all_labels, all_preds, average='binary', pos_label=pos)
    train_recall = recall_score(all_labels, all_preds, average='binary', pos_label=pos)
    train_f1 = f1_score(all_labels, all_preds, average='binary', pos_label=pos)

    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    train_precisions.append(train_precision)
    train_recalls.append(train_recall)
    train_f1s.append(train_f1)

    #$ --------------------------------------------- Validation Loop ---------------------------------------------
    model.eval()
    val_running_loss, val_correct, val_total = 0.0,0 ,0
    val_labels_all, val_preds_all = [], []

    with torch.no_grad():

        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]"):
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)

            labels = labels.float().unsqueeze(1)
            loss = criterion(outputs, labels)

            probs = torch.sigmoid(outputs)
            predicted = (probs > 0.5).long()

            val_running_loss += loss.item() * inputs.size(0)
            val_correct += (predicted == labels.long()).sum().item()
            val_total += labels.size(0)

            val_labels_all.extend(labels.cpu().numpy())
            val_preds_all.extend(predicted.cpu().numpy())

    val_loss = val_running_loss / len(val_dataset)
    val_acc = val_correct / val_total
    val_precision = precision_score(val_labels_all, val_preds_all, average='binary', pos_label=pos)
    val_recall = recall_score(val_labels_all, val_preds_all, average='binary', pos_label=pos)
    val_f1 = f1_score(val_labels_all, val_preds_all, average='binary', pos_label=pos)

    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    val_precisions.append(val_precision)
    val_recalls.append(val_recall)
    val_f1s.append(val_f1)

       # Save metrics after each epoch
    metrics = {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "train_accuracies": train_accuracies,
        "val_accuracies": val_accuracies,
        "train_precisions": train_precisions,
        "train_recalls": train_recalls,
        "train_f1s": train_f1s,
        "val_precisions": val_precisions,
        "val_recalls": val_recalls,
        "val_f1s": val_f1s,
    }
    with open("training_metrics.json", "w") as f:
        json.dump(metrics, f)



    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.5f}, Acc: {train_acc*100:.2f}%, Precision: {train_precision:.4f}, Recall: {train_recall:.4f}, F1: {train_f1:.4f}")
    print(f"Val   Loss: {val_loss:.5f}, Acc: {val_acc*100:.2f}%, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}")
    print("-----------------------------------------------------------")

end_time = timer() # end timing
training_time = print_train_time(start_time,end_time, device)

# Section 3

In [ ]:
class_names = train_dataset.classes
print(f"Class names: {class_names}")

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(train_accuracies, label="Train Accuracy")
plt.plot(val_accuracies, label="Val Accuracy")
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("Loss_MRI_RN18_M1_TL.png")

plt.show()

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(train_precisions, label="Train Precision")
plt.plot(val_precisions, label="Val Precision")
plt.xlabel('Epochs')
plt.ylabel('Precision')
plt.title('Precision')
plt.legend()
plt.grid(True)
#

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8,6))
plt.plot(train_recalls, label="Train Recall")
plt.plot(val_recalls, label="Val Recall")
plt.xlabel('Epochs')
plt.ylabel('Recall')
plt.title('Recall')
plt.legend()
plt.grid(True)


plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(train_f1s, label="Train F1")
plt.plot(val_f1s, label="Val F1")
plt.xlabel('Epochs')
plt.ylabel('F1 Score')
plt.title('F1 Score')
plt.legend()
plt.grid(True)

plt.tight_layout()

plt.show()

In [ ]:
save_path_state_dict = "state_dict_MRI_RN18.pth"
save_path_entire_model = "entire_MRI_RN18.pth"

torch.save(model.state_dict(), save_path_state_dict)
print(f"Model state dictionary saved to {save_path_state_dict}")

torch.save(model, save_path_entire_model)
print(f"Entire model saved to {save_path_entire_model}")